[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-09-debugging-observability.ipynb#scrollTo=11a2b3c4)

---
# Day 9 · Debugging and Observability — Dashboard, Timeline, and Logs
**certified-journeys / ray-certified** · Review Day

> **Goal for today:** Use the Ray Dashboard, `ray.timeline()`, structured logging with runtime context, and the `ray.util.state` API to diagnose performance bottlenecks and introspect live cluster state.


In [ ]:
%pip install -q 'ray[default]'


## Concept 1 · The Ray Dashboard

The Ray Dashboard is a web UI that auto-starts when you call `ray.init()`. It gives you live visibility into:

| Tab | What you see |
|---|---|
| **Cluster** | Node list, CPU/GPU/memory per node, object store usage |
| **Jobs** | All submitted jobs; status, logs, runtime env |
| **Actors** | All live actors; name, class, resource usage, PID |
| **Tasks** | Recent finished/running tasks; duration, node |
| **Metrics** | Prometheus-backed charts (requires Prometheus sidecar in prod) |
| **Logs** | Aggregated stdout/stderr from all workers |

By default the dashboard runs at `http://127.0.0.1:8265`. In Colab you can access it via `ray.init()`'s printed URL, or tunnel it externally.


In [ ]:
import ray
import time

# include_dashboard=True ensures the dashboard process starts
ctx = ray.init(ignore_reinit_error=True, include_dashboard=True)

# ray.init() returns a RayContext — the dashboard_url is available in it
print("Ray version      :", ray.__version__)
print("Dashboard URL    :", ctx.dashboard_url if hasattr(ctx, 'dashboard_url') else 'http://127.0.0.1:8265')
print("Cluster resources:", ray.cluster_resources())


### What just happened?
- **`ray.init()`** started a local single-node Ray cluster — head node, Raylet, GCS, and object store — plus the dashboard process.
- **`ray.cluster_resources()`** returns the total available resources (CPU, memory, GPU if any) across all nodes.
- In a **production multi-node cluster** (Kubernetes, YARN, or bare metal), `ray.init(address='auto')` connects to the existing cluster rather than spawning one.
- The Dashboard Metrics tab integrates with **Prometheus + Grafana** for production-grade monitoring; it requires `ray[default]` and a running Prometheus scrape config.


## Concept 2 · `ray.timeline()` — Chrome Trace Profiling

`ray.timeline()` captures a **distributed timeline** of every task that ran in the cluster. The output is a JSON file compatible with Chrome's `chrome://tracing` viewer and Perfetto UI.

### Reading the timeline

Each row is a worker thread or actor. Each box is a task. You can spot:

| Pattern | Meaning | Fix |
|---|---|---|
| Wide gap before task starts | Workers unavailable (resource starvation) | Check `num_cpus` vs available CPUs |
| Tasks serialised (no overlap) | Dependencies not parallelised | Restructure to maximise concurrency |
| Short tasks, long gaps | Scheduling overhead dominates | Batch small tasks |
| Large memory objects | Object transfer latency | Use `ray.put()` once; share refs |


In [ ]:
import numpy as np

# ── Run a parallel workload to generate timeline events ──────────────────────

@ray.remote
def heavy_compute(seed: int, size: int = 500_000) -> float:
    """Simulate CPU-bound work: generate a large array and compute its norm."""
    rng = np.random.default_rng(seed)
    arr = rng.standard_normal(size)
    # Intentionally sequential operations to generate meaningful trace events
    result = float(np.linalg.norm(arr))
    return result

@ray.remote
def aggregate(values: list) -> dict:
    """Collect results from parallel workers."""
    return {
        "count": len(values),
        "mean" : float(np.mean(values)),
        "min"  : float(np.min(values)),
        "max"  : float(np.max(values)),
    }

# Launch 8 parallel tasks, then aggregate
print("Launching parallel tasks...")
futures = [heavy_compute.remote(i) for i in range(8)]
results = ray.get(futures)

summary = ray.get(aggregate.remote(results))
print("Aggregate summary:", summary)


### What just happened?
- Eight `heavy_compute` tasks ran in parallel across available workers, each doing CPU-intensive numpy operations.
- The `aggregate` task depended on all 8 results — it ran only after `ray.get(futures)` resolved all of them.
- This fan-out → fan-in pattern is exactly what makes Ray timelines informative: you can see the parallel lanes and the synchronisation barrier.


In [ ]:
import os, json

# Capture the timeline — this is a Chrome-compatible JSON trace
timeline_path = "/tmp/ray_timeline.json"
ray.timeline(filename=timeline_path)

# Inspect the trace structure (first 3 events)
with open(timeline_path, "r") as f:
    trace_data = json.load(f)

# The trace can be either a list or a dict with a 'traceEvents' key
events = trace_data if isinstance(trace_data, list) else trace_data.get("traceEvents", [])
print(f"Total trace events: {len(events)}")
print("\nSample events:")
for ev in events[:3]:
    print(f"  name={ev.get('name','?'):<30} ph={ev.get('ph','?')}  ts={ev.get('ts',0):.0f} μs")

print(f"\nTimeline saved to: {timeline_path}")
print("Open in Perfetto: https://ui.perfetto.dev  (drag-and-drop the file)")
print("Or in Chrome    : navigate to chrome://tracing and click Load")


### What just happened?
- **`ray.timeline(filename=...)`** writes a Chrome Trace Event format JSON file — every task is a duration event (`ph: 'X'`) with start time and duration in microseconds.
- The `name` field maps to your function name; the `tid` (thread ID) determines which row the event appears on in the viewer.
- **Wide gaps between task submission and task start** are the most common bottleneck: they mean workers were unavailable. Look for `num_cpus` mismatches or resource contention.
- In production, use [Perfetto UI](https://ui.perfetto.dev) — it handles multi-MB trace files better than `chrome://tracing`.


## Concept 3 · Structured Logging with `ray.runtime_context`

Inside a remote task or actor, `ray.runtime_context.get_runtime_context()` gives you metadata about the current execution context:

```python
ctx = ray.runtime_context.get_runtime_context()
ctx.get_job_id()     # Job ID string
ctx.get_task_id()    # Task ID string (unique per task invocation)
ctx.get_node_id()    # Node ID where this task is running
ctx.get_worker_id()  # Worker process ID
```

Use these as **structured log fields** so you can correlate log lines with specific tasks or actors in the Dashboard Logs tab.


In [ ]:
import logging

@ray.remote
def instrumented_task(x: int) -> dict:
    """Task that emits structured log lines with runtime context."""
    ctx = ray.runtime_context.get_runtime_context()

    # Build a structured log prefix with context identifiers
    log_prefix = {
        "job_id"  : ctx.get_job_id(),
        "task_id" : str(ctx.get_task_id())[:8],   # truncate for readability
        "node_id" : str(ctx.get_node_id())[:8],
    }

    # Structured print — in production, swap for logging.info(json.dumps(...))
    print(f"[START] x={x}  context={log_prefix}")
    result = x ** 2
    print(f"[END]   x={x}  result={result}  context={log_prefix}")

    return {"input": x, "output": result, "context": log_prefix}

# Run 3 tasks and collect their structured logs
futures = [instrumented_task.remote(i) for i in range(3)]
outputs = ray.get(futures)
print("\nTask outputs:")
for o in outputs:
    print(f"  input={o['input']}  output={o['output']}  job={o['context']['job_id']}")


### What just happened?
- **`get_runtime_context()`** is only valid *inside* a remote function or actor method — it returns `None` in the driver.
- The `job_id`, `task_id`, and `node_id` fields tie log lines to specific executions, making it possible to trace a single request across multiple tasks.
- Ray automatically captures all `print()` output from workers and makes it available in the Dashboard Logs tab, filtered by job or worker.
- For production, replace `print()` with Python's `logging` module configured to emit JSON — tools like Datadog, Splunk, or Loki can then parse and index these fields.


## Concept 4 · `ray.util.state` — Programmatic Cluster Introspection

The `ray.util.state` module lets you query live cluster state from Python — no dashboard needed. This is useful for:
- **Automated health checks** in CI
- **Dynamic resource management** (e.g., wait until N actors are ready)
- **Debugging** stuck tasks or leaked actors

| Function | Returns |
|---|---|
| `list_tasks()` | All recent tasks with status, duration, node |
| `list_actors()` | All live actors with class name, pid, state |
| `list_objects()` | Object store entries with size and reference counts |
| `list_nodes()` | Nodes with resource totals and availability |
| `get_task(task_id)` | Detailed info for one task |


In [ ]:
from ray.util.state import list_tasks, list_actors, list_nodes

# ── Inspect tasks ─────────────────────────────────────────────────────────────
tasks = list_tasks()
print(f"Recent tasks ({len(tasks)} total):")
for t in tasks[:5]:
    # Each TaskState has: task_id, name, state, duration_ms, node_id
    duration_ms = getattr(t, 'duration_ms', None) or 0
    print(f"  {t.name:<35} state={t.state:<12} dur={duration_ms:.0f}ms")

print()

# ── Inspect nodes ─────────────────────────────────────────────────────────────
nodes = list_nodes()
print(f"Cluster nodes ({len(nodes)} total):")
for n in nodes:
    total_cpus = n.resources_total.get('CPU', 0)
    state = getattr(n, 'state', 'ALIVE')
    print(f"  node_id={str(n.node_id)[:12]}...  CPUs={total_cpus}  state={state}")


### What just happened?
- **`list_tasks()`** queries the GCS (Global Control Store) — the distributed metadata store — and returns `TaskState` objects with rich task metadata.
- Tasks in state `FINISHED` are from completed calls; `RUNNING` means currently executing; `PENDING_ARGS_AVAIL` means waiting for input object refs to resolve.
- **`list_nodes()`** is especially useful in autoscaling clusters to check current node count before submitting large jobs.
- These APIs are also available from the CLI: `ray list tasks`, `ray list actors`, `ray list nodes`.


## Concept 5 · Memory Debugging and Object Spilling

Ray stores objects in a shared-memory **object store** (backed by Plasma). When the object store fills up, Ray **spills** objects to disk (or S3 in production) to free memory.

### Signs of memory pressure

```
ray.exceptions.ObjectStoreFullError: Failed to put object of size X bytes
WARNING  spillin_to_disk: Spilled 1.5 GiB, 2.3 GiB remaining
```

### Debugging memory

```python
# See current object store usage
import ray
mem_info = ray.internal.internal_api.memory_summary(stats_only=True)
print(mem_info)
```

### Configuring spilling

```python
ray.init(
    object_store_memory=2 * 1024**3,  # 2 GB object store
    _system_config={
        "object_spilling_config": json.dumps({
            "type": "filesystem",
            "params": {"directory_path": "/tmp/spill"}
        })
    }
)
```

For production use S3 spilling:
```python
{"type": "smart_open", "params": {"uri": "s3://bucket/spill/"}}
```


## Concept 6 · Actor Introspection

Named actors can be found at runtime using `ray.get_actor(name)`. Combined with `list_actors()`, this enables:
- **Health polling**: check if a critical actor is still alive
- **Graceful shutdown**: call a cleanup method before terminating
- **Debugging deadlocks**: inspect an actor's pending queue length

```python
# Create a named actor
@ray.remote
class Counter:
    def __init__(self): self.n = 0
    def increment(self): self.n += 1
    def get(self): return self.n

counter = Counter.options(name="global-counter", lifetime="detached").remote()

# Retrieve it by name from anywhere in the cluster
same_counter = ray.get_actor("global-counter")
```


In [ ]:
# ── Named actor + state introspection ────────────────────────────────────────

@ray.remote
class Accumulator:
    """Simple stateful actor — accumulates values."""
    def __init__(self):
        self.values = []

    def add(self, v):
        self.values.append(v)
        return len(self.values)

    def total(self):
        return sum(self.values)

    def stats(self):
        ctx = ray.runtime_context.get_runtime_context()
        return {
            "count" : len(self.values),
            "total" : sum(self.values),
            "actor_id": str(ctx.get_actor_id())[:12],
        }

# Launch with a name so we can find it via list_actors or get_actor
acc = Accumulator.options(name="my-accumulator").remote()
ray.get([acc.add.remote(i) for i in range(10)])
print("Actor stats:", ray.get(acc.stats.remote()))

# List all live actors — should include my-accumulator
actors = list_actors()
print(f"\nLive actors ({len(actors)} total):")
for a in actors:
    print(f"  name={getattr(a, 'name', '?'):<25} class={getattr(a, 'class_name', '?'):<20} state={a.state}")


### What just happened?
- **`Accumulator.options(name=...)`** registers the actor with the GCS under a stable name — any worker in the cluster can find it via `ray.get_actor("my-accumulator")`.
- **`list_actors()`** returns actor metadata from the GCS without needing a handle — useful for monitoring dashboards and health checks.
- The `actor_id` from `get_runtime_context()` lets you cross-reference log lines with the entry in `list_actors()` output.
- Actors in `DEAD` state are removed from the list after a grace period; actors in `RESTARTING` are being respawned by the fault-tolerance system.


## Concept 7 · Common Debugging Patterns — Quick Reference

| Symptom | Tool | What to look for |
|---|---|---|
| Tasks queue but don't start | `ray.cluster_resources()` | Available CPUs < requested CPUs |
| Memory errors / OOM | Dashboard Metrics → Node Memory | Object store usage > 70% |
| Actor unresponsive | `list_actors()` | State = `RESTARTING` or `DEAD` |
| Slow pipeline | `ray.timeline()` | Wide gaps before tasks start |
| Can't find a log line | Dashboard Logs | Filter by `job_id` or `node_id` |
| Object store pressure | `list_objects()` | Large objects with ref_count > 1 |

### The observability stack in production

```
Ray workers → stdout/stderr → Ray log aggregator → Loki / Datadog
Ray metrics → Prometheus exporter (port 8080) → Grafana dashboards
ray.timeline() → Perfetto UI (drag-and-drop JSON)
ray.util.state → Custom health check scripts / alerting
```


In [ ]:
# Challenge: Build a resource-aware task submitter
#
# Goal: write a function submit_if_resources_available(task_fn, *args, min_cpus=1.0)
# that:
#   1. Calls ray.available_resources() to check current free CPUs
#   2. If available CPUs >= min_cpus: submit the task and return the future
#   3. If not: print a warning and return None
#
# Then:
#   - Define a @ray.remote def slow_task(x) that sleeps 0.1s and returns x**2
#   - Submit 4 tasks using submit_if_resources_available
#   - Collect non-None futures with ray.get() and print results
#
# Bonus: after submitting, call ray.timeline('/tmp/challenge_timeline.json')
#        and print how many trace events were captured

# Your solution here:
# def submit_if_resources_available(task_fn, *args, min_cpus=1.0):
#     ...


In [ ]:
# Cleanup
ray.shutdown()
print("Ray shut down.")


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| Ray Dashboard | Auto-starts at port 8265; tabs: Cluster, Jobs, Actors, Tasks, Metrics, Logs |
| `ray.timeline()` | Chrome Trace JSON; open in Perfetto or chrome://tracing |
| Wide pre-task gap | Workers unavailable — check CPU requests vs cluster capacity |
| `get_runtime_context()` | Returns job_id, task_id, node_id from inside a remote function |
| `list_tasks()` | Query all recent tasks with status, duration, node from Python |
| `list_actors()` | Query all live actors; find dead/restarting ones programmatically |
| Object spilling | Ray spills from object store to disk when memory pressure hits |
| Named actors | `options(name=...)` + `ray.get_actor()` for cluster-wide lookup |

> **Tip:** A wide gap between task submission and task start in the timeline means workers are unavailable — check resource requests vs available CPUs. The Dashboard Metrics tab integrates with Prometheus for production monitoring.

---
## What's next
**Day 10** → Capstone — build an end-to-end distributed ML pipeline: Ray Data → Ray Train → Ray Tune → Ray Serve.

Mark Day 9 complete in your [tracker](../index.html).
